# 迁移学习示例：ResNet18 特征 → sklearn → SSME-Lite

这条线和官方 PneumoniaMNIST checkpoint 不是一回事。官方模型已经在这份数据上训练过；这里的 backbone 只使用 ImageNet 预训练权重，并且全程冻结。

$$
\text{胸片}
\rightarrow
\text{冻结的 ImageNet ResNet18}
\rightarrow
\text{sklearn 分类头}
\rightarrow
\text{SSME-Lite}.
$$

分类头只在官方 **train** split 上拟合。SSME 只看官方 **test** split：其中少量标签可见，其余标签留到评估时才使用。验证集不参与选模。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "transfer_pneumonia.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "examples"))

import numpy as np
import pandas as pd
from ssme_lite import PredictionMatrixGenerator, SSMEEstimator, SemiSupervisedSplit
from transfer_pneumonia import FEATURE_FILE, TRANSFER_SUBSETS, cached_features, fit_heads

## 1. 数据

PneumoniaMNIST 把儿科胸片收成 28×28 灰度图，标签是 normal / pneumonia。官方 test 只有 624 张，所以估计协议用 20 个标签和 400 个无标签，而不是论文里的 1000 个无标签。划分由 `SemiSupervisedSplit` 固定。

In [2]:
images = np.load(ROOT / "data" / "pneumoniamnist" / "pneumoniamnist.npz")
print({key: images[key].shape for key in images.files})
for split in ("train", "test"):
    y = images[f"{split}_labels"].reshape(-1)
    print(split, "normal", int((y == 0).sum()), "pneumonia", int((y == 1).sum()))

{'train_images': (4708, 28, 28), 'val_images': (524, 28, 28), 'test_images': (624, 28, 28), 'train_labels': (4708, 1), 'val_labels': (524, 1), 'test_labels': (624, 1)}
train normal 1214 pneumonia 3494
test normal 234 pneumonia 390


## 2. 冻结的 ResNet18 特征

去掉最后的分类层，得到 512 维特征。28×28 灰度图先复制成 3 通道，双线性放大到 224，再做 ImageNet 归一化。放大不会恢复原始胸片分辨率；这个例子演示的是迁移流程，不是把肺炎分类做到最好。

In [3]:
X_train, y_train, X_test, y_test = cached_features(device="auto")
print(FEATURE_FILE)
print("train features", X_train.shape, "test features", X_test.shape)

/Users/jason/Desktop/研一上/机器学习与 Python 应用/data/pneumoniamnist/resnet18_features.npz
train features (4708, 512) test features (624, 512)


## 3. 同一组特征上的多个分类头

Logistic Regression、Random Forest 和 MLP 是三条不同的决策边界。另外加上 Extra Trees 和 HistGradientBoosting，凑成 5 个有差异的 `predict_proba`，后面才能做模型数量消融。超参数事先写死，没有用验证集挑选。

In [4]:
models = fit_heads(X_train, y_train, seed=0)
generator = PredictionMatrixGenerator(models)
scores = generator.fit_transform(X_test)
print("models", generator.model_names_)
print("scores", scores.shape, "classes", generator.classes_)

models ['logistic', 'random_forest', 'mlp', 'extra_trees', 'hist_gradient_boosting']
scores (624, 5) classes [0 1]


## 4. 接到 SSME-Lite

`SemiSupervisedSplit` 把 test split 分成三份：SSME 看得见的标签、SSME 只看得见预测的无标签样本，以及完全不参与拟合的 held-out。估计器消费的是 `N × M` 的正类概率，不是原始图像。

In [5]:
splitter = SemiSupervisedSplit(n_labeled=20, n_unlabeled=400, random_state=0)
parts = splitter.indices(len(y_test))
partial = splitter.partial_labels(y_test, parts)
print({name: len(index) for name, index in parts.items()})
print("visible labels", int((partial >= 0).sum()), "hidden in the estimation pool", int((partial < 0).sum()))

estimator = SSMEEstimator(max_iter=100, early_stopping=False, random_state=0)
estimator.fit(scores[parts["estimation"]], partial, model_names=generator.model_names_)
report = estimator.report(n_draws=200, title="PneumoniaMNIST transfer · ResNet18 features", primary_metric="auc")
report.ranking("auc")

{'labeled': 20, 'unlabeled': 400, 'estimation': 420, 'heldout': 204}
visible labels 20 hidden in the estimation pool 400


,model,metric,estimate,label_std,mc_standard_error,interval_low,interval_high,valid_draws,rank
9,mlp,auc,0.982102,0.004699,0.000332,0.972910,0.990757,200,1.0
1,logistic,auc,0.974878,0.005089,0.000360,0.963375,0.982984,200,2.0
17,hist_gradient_boosting,auc,0.964725,0.006858,0.000485,0.951855,0.977340,200,3.0
5,random_forest,auc,0.953273,0.007850,0.000555,0.935865,0.966030,200,4.0
13,extra_trees,auc,0.950001,0.007711,0.000545,0.934779,0.962394,200,5.0


In [6]:
report.table.pivot(index="model", columns="metric", values="estimate").round(3)

metric,accuracy,auc,auprc,ece
model,,,,
extra_trees,0.849,0.950,0.975,0.104
hist_gradient_boosting,0.875,0.965,0.981,0.089
logistic,0.889,0.975,0.987,0.099
mlp,0.897,0.982,0.991,0.088
random_forest,0.860,0.953,0.976,0.092


上面是一次划分上的估计，还不是 ground truth。重复实验（10 个 seed，`n_l = 20, 50, 100`，以及 2–5 个分类头）由 `python examples/transfer_pneumonia.py` 写入 `results/pneumonia_transfer/`。误差是 `|估计 − held-out 真值|` 的跨 seed 均值。嵌套子集是事先指定的：

In [7]:
summary = ROOT / "results" / "pneumonia_transfer" / "nl20_nu400_m5" / "summary.csv"
print(TRANSFER_SUBSETS)
if summary.exists():
    table = pd.read_csv(summary)
    view = table[table.target == "heldout"][["method", "metric", "mae", "mae_ci_low", "mae_ci_high"]]
    display(view.round(4))
else:
    print("重复实验还没跑。执行 python examples/transfer_pneumonia.py")

{'2': ['logistic', 'random_forest'], '3': ['logistic', 'random_forest', 'mlp'], '4': ['logistic', 'random_forest', 'mlp', 'extra_trees'], '5': ['logistic', 'random_forest', 'mlp', 'extra_trees', 'hist_gradient_boosting']}


,method,metric,mae,mae_ci_low,mae_ci_high
8,labeled_only,accuracy,0.0711,0.0461,0.0962
9,labeled_only,auc,0.0475,0.0326,0.0625
10,labeled_only,auprc,0.0427,0.0319,0.0534
11,labeled_only,ece,0.0643,0.0421,0.0864
12,ssme,accuracy,0.0416,0.0250,0.0583
13,ssme,auc,0.0389,0.0222,0.0555
14,ssme,auprc,0.0405,0.0181,0.0628
15,ssme,ece,0.0396,0.0263,0.0528
